In [ ]:
using Pkg
Pkg.instantiate()

   Installed Libmount_jll ───────────────── v2.41.0+0
   Installed libfdk_aac_jll ─────────────── v2.0.3+0
   Installed GR_jll ─────────────────────── v0.73.14+0
   Installed JpegTurbo_jll ──────────────── v3.1.1+0
   Installed x265_jll ───────────────────── v3.5.0+0
   Installed FFTW ───────────────────────── v1.8.1
   Installed Preferences ────────────────── v1.4.3
   Installed MutableArithmetics ─────────── v1.6.4
   Installed LoggingExtras ──────────────── v1.1.0
   Installed Opus_jll ───────────────────── v1.3.3+0
   Installed InlineStrings ──────────────── v1.4.3
   Installed Unitful ────────────────────── v1.22.0
   Installed StaticArrays ───────────────── v1.9.13
   Installed NearestNeighbors ───────────── v0.4.21
   Installed Xorg_xcb_util_wm_jll ───────── v0.4.1+1
   Installed Xorg_xcb_util_image_jll ────── v0.4.0+1
   Installed Cairo_jll ──────────────────── v1.18.4+0
   Installed OpenSSL ────────────────────── v1.4.3
   Installed HTTP ───────────────────────── v1.10.16
   I

In [1]:
# 必要なパッケージを読み込む
using DelimitedFiles, DataFrames,  Plots
Plots.default(fontfamily="IPAexGothic")
# 前提として、以下のファイルが既に存在している
include("/workspaces/inulab_julia_devcontainer/src/calc_IPW.jl")          # リグレット計算の基本関数
include("/workspaces/inulab_julia_devcontainer/src/file_operate.jl")      # ファイル操作用関数
# 1. データの読み込み
function load_data(method_name="simp", repeat_num=1, criteria_num=6)
    # 効用値行列の読み込み
    utility_data = read_utility_value()[5]
    
    # 区間重要度の読み込み
    method_weights = read_method_weights(method_name, repeat_num, criteria_num)[5]
    
    return utility_data, method_weights
end

# 2. 最適なtの範囲を計算
function calculate_t_range(method_weights)
    t_range = find_optimal_trange(method_weights.L, method_weights.R)
    # println("最適t範囲: [", t_range[1], ", ", t_range[2], "]")
    return t_range
end

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `/workspaces/inulab_julia_devcontainer/Project.toml`
  No Changes to `/workspaces/inulab_julia_devcontainer/Manifest.toml`


calculate_t_range (generic function with 1 method)

In [ ]:
repeat_num = 100;##ひとつの真の重要度の組に対するプログラムの繰り返し回数(100回)
criteria_num = 6;#代替案の個数（6個）
utility_matrix_num = 100;#効用値行列の個数（100個）
weight_type = ["A","B","C","D","E"];#５タイプの真の重要度区間
method_name_list = [
    "/EV", "/GM", "/WMIN", "/DMIN", "/MMRW", "/MMRD",
    "/E-MMRW", "/G-MMRW", "/E-MMRD", "/G-MMRD", "/AMRW", "/AMRD", "/E-AMRW",
    "/G-AMRW", "/E-AMRD", "/G-AMRD", "/eAMRw", "/eAMRwc", "/eMMRw", "/eMMRwc",
    "/gAMRw", "/gAMRwc", "/gMMRw", "/gMMRwc", "/AMRwc", "/MMRwc",
    "/eAMRd", "/eAMRdc", "/eMMRd", "/eMMRdc", "/gAMRd", "/gAMRdc", "/gMMRd", "/gMMRdc",
]
utility_methods = ["u1","u2"]
#効用値読み取り5行6列ごとに2次元配列としてデータが格納
utility_data = read_utility_value(utility_methods[1])
for type in weight_type     #A-Eでループ
    #真の区間重要度を取得 trueW.L,trueW.Rとして左と右が取得できる
    trueW= read_true_weights(type)
    s_range_true = find_optimal_trange(trueW.L, trueW.R)
    @threads for method in method_name_list
        tid = threadid()
        println("$type:$method を処理中, スレッドID: ", tid)
        filepath = "/workspaces/inulab_julia_devcontainer/data/a3/regret/u1/N=6/$type/$method/u1_minimax_regret_$repeat_num.csv"
        methodW = read_method_weights(type * method, repeat_num,criteria_num)
        open(filepath, "w") do io
            for utl_num in 1:utility_matrix_num
                for Repeat_time in 1:repeat_num
                    t_range = calculate_t_range(methodW[Repeat_time])
                    
                end
            end
        end
    end
end

In [5]:
using CSV
using DataFrames
using Plots

# ========== Paths ==========
base_dir   = "/workspaces/inulab_julia_devcontainer"
input_path = base_dir * "/results/trange_tidy/trange_tidy_u1u2_all_methods.csv"
out_dir    = base_dir * "/results/trange_tidy/plots_t_scatter"

mkpath(out_dir)

# 対象 4 手法
selected_methods = ["EV", "eAMRw", "MMRW", "gAMRw"]

# Method 列の "/EV" → "EV" みたいな正規化
normalize_method(name::AbstractString) = startswith(name, "/") ? name[2:end] : String(name)

function plot_t_scatter_for_slide(
    weight_type::String = "A",
    utility_case::String = "u1",
)
    println("Loading: $input_path")
    df = CSV.read(input_path, DataFrame)

    # Method を正規化
    df.MethodShort = normalize_method.(df.Method)

    # 条件でフィルタ
    sub = filter(row ->
        row.WeightType == weight_type &&
        row.UtilityCase == utility_case &&
        (row.MethodShort in selected_methods),
        df,
    )

    if nrow(sub) == 0
        @warn "No rows matched for WeightType=$weight_type, UtilityCase=$utility_case"
        return
    end

    # 手法ごとに 1 枚ずつプロット
    for method in selected_methods
        sub_m = sub[sub.MethodShort .== method, :]

        if nrow(sub_m) == 0
            @warn "No data for method=$method (WeightType=$weight_type, UtilityCase=$utility_case)"
            continue
        end

        # 対応関係：
        #   L: (x = tL_est, y = sL_true)
        #   U: (x = tR_est, y = sR_true)
        xL = sub_m.tL_est
        yL = sub_m.sL_true
        xU = sub_m.tR_est
        yU = sub_m.sR_true

        # 軸範囲を決める（L/U/true/est 全部の min/max）
        all_x = vcat(xL, xU)
        all_y = vcat(yL, yU)
        xmin = minimum(all_x)
        xmax = maximum(all_x)
        ymin = minimum(all_y)
        ymax = maximum(all_y)

        lo = min(xmin, ymin) - 0.05
        hi = max(xmax, ymax) + 0.05

        lo = min(xmin, ymin)
        hi = max(xmax, ymax)

        # 散布図
        p = scatter(
            xL, yL;
            label="L: estimated vs true",
            color=:orange,
            markersize=3,
            grid=true,
        )
        scatter!(
            p, xU, yU;
            label="U: estimated vs true",
            color=:blue,
            markersize=3,
        )

        # 完全一致線 y = x
        xs = range(lo, hi; length=100)
        plot!(
            p, xs, xs;
            label="y = x (perfect)",
            color=:black,
            linestyle=:dash,
            linewidth=1,
        )

        xlims!(p, lo, hi)
        ylims!(p, lo, hi)
        xlabel!(p, "Estimated t (L / U)")
        ylabel!(p, "True t (L / U)")
        title!(p, "Method: $method  (Weight=$weight_type, Utility=$utility_case)")
        plot!(p; aspect_ratio=:equal)

        out_file = joinpath(
            out_dir,
            "t_scatter_$(method)_Weight$(weight_type)_$(utility_case).png",
        )
        savefig(p, out_file)
        println("Saved plot:", out_file)
    end
end

# 実行例：WeightType A, UtilityCase u1
plot_t_scatter_for_slide("A", "u1")

# 必要なら：
# plot_t_scatter_for_slide("A", "u2")
# plot_t_scatter_for_slide("B", "u1") など


Loading: /workspaces/inulab_julia_devcontainer/results/trange_tidy/trange_tidy_u1u2_all_methods.csv
Saved plot:/workspaces/inulab_julia_devcontainer/results/trange_tidy/plots_t_scatter/t_scatter_EV_WeightA_u1.png
Saved plot:/workspaces/inulab_julia_devcontainer/results/trange_tidy/plots_t_scatter/t_scatter_eAMRw_WeightA_u1.png
Saved plot:/workspaces/inulab_julia_devcontainer/results/trange_tidy/plots_t_scatter/t_scatter_MMRW_WeightA_u1.png
Saved plot:/workspaces/inulab_julia_devcontainer/results/trange_tidy/plots_t_scatter/t_scatter_gAMRw_WeightA_u1.png


In [4]:
using CSV
using DataFrames
using Statistics

# -------------------------
# 小さい値を 0 とみなすヘルパー
# -------------------------
const EPS_SMALL = 1e-10

"""
    clip_small(x; eps = EPS_SMALL)

|x| < eps のとき 0.0 に丸める。
"""
clip_small(x; eps = EPS_SMALL) = (abs(x) < eps ? 0.0 : x)

# -------------------------
# ヘルパー: t範囲 → 中心・幅
# -------------------------
"""
    summarize_trange(t_range)

t_range = (tL, tR) または 2 要素ベクトルを渡すと、
(tL, tR, mid, width) を返す。
"""
function summarize_trange(t_range)
    tL = t_range[1]
    tR = t_range[2]
    mid = (tL + tR) / 2
    width = tR - tL
    return tL, tR, mid, width
end

# -------------------------
# 入力設定
# -------------------------
repeat_num        = 100   # 1 つの真の重要度の組に対する繰り返し
criteria_num      = 6     # 評価基準数
utility_matrix_num = 100  # 効用値行列の個数

weight_types = ["A","B","C","D","E"]

method_name_list = [
    "/EV", "/GM", "/WMIN", "/DMIN", "/MMRW", "/MMRD",
    "/E-MMRW", "/G-MMRW", "/E-MMRD", "/G-MMRD", "/AMRW", "/AMRD", "/E-AMRW",
    "/G-AMRW", "/E-AMRD", "/G-AMRD", "/eAMRw", "/eAMRwc", "/eMMRw", "/eMMRwc",
    "/gAMRw", "/gAMRwc", "/gMMRw", "/gMMRwc", "/AMRwc", "/MMRwc",
    "/eAMRd", "/eAMRdc", "/eMMRd", "/eMMRdc", "/gAMRd", "/gAMRdc", "/gMMRd", "/gMMRdc",
]

utility_methods = ["u1","u2"]  # 効用ケース

# -------------------------
# メイン集計関数
# -------------------------
function collect_trange_summary(output_path::String)

    df = DataFrame(
        WeightType     = String[],
        Method         = String[],
        UtilityCase    = String[],
        UtilityIndex   = Int[],
        RepeatIndex    = Int[],
        sL_true        = Float64[],
        sR_true        = Float64[],
        s_mid_true     = Float64[],
        s_width_true   = Float64[],
        tL_est         = Float64[],
        tR_est         = Float64[],
        t_mid_est      = Float64[],
        t_width_est    = Float64[],
    )

    for ucase in utility_methods
        println("=== UtilityCase: $ucase ===")

        utility_data = read_utility_value(ucase)

        for wt in weight_types
            println("  WeightType: $wt")

            trueW = read_true_weights(wt)
            s_range_true = find_optimal_trange(trueW.L, trueW.R)
            sL_true, sR_true, s_mid_true, s_width_true = summarize_trange(s_range_true)

            # 小さい値はここで丸めておく
            sL_true      = clip_small(sL_true)
            sR_true      = clip_small(sR_true)
            s_mid_true   = clip_small(s_mid_true)
            s_width_true = clip_small(s_width_true)

            for method in method_name_list
                println("    Method: $method")

                methodW_all = read_method_weights(wt * method, repeat_num, criteria_num)

                for utl_idx in 1:utility_matrix_num
                    for rep in 1:repeat_num
                        # ★ 今の想定どおり「rep だけでよい」場合
                        estW = methodW_all[rep]

                        t_range = find_optimal_trange(estW.L, estW.R)
                        tL_est, tR_est, t_mid_est, t_width_est = summarize_trange(t_range)

                        # 推定側も小さい値は丸める
                        tL_est      = clip_small(tL_est)
                        tR_est      = clip_small(tR_est)
                        t_mid_est   = clip_small(t_mid_est)
                        t_width_est = clip_small(t_width_est)

                        push!(df, (
                            wt,              # WeightType
                            method,          # Method
                            ucase,           # UtilityCase
                            utl_idx,         # UtilityIndex
                            rep,             # RepeatIndex
                            sL_true,
                            sR_true,
                            s_mid_true,
                            s_width_true,
                            tL_est,
                            tR_est,
                            t_mid_est,
                            t_width_est,
                        ))
                    end
                end
            end
        end
    end

    mkpath(dirname(output_path))
    CSV.write(output_path, df)
    println("Saved tidy t-range summary to: $output_path")
    return df
end

# 実行例
output_csv = "./results/trange_tidy/trange_tidy_u1u2_all_methods.csv"
df_trange = collect_trange_summary(output_csv)


=== UtilityCase: u1 ===
  WeightType: A
    Method: /EV
    Method: /GM
    Method: /WMIN
    Method: /DMIN
    Method: /MMRW
    Method: /MMRD
    Method: /E-MMRW
    Method: /G-MMRW
    Method: /E-MMRD
    Method: /G-MMRD
    Method: /AMRW
    Method: /AMRD
    Method: /E-AMRW
    Method: /G-AMRW
    Method: /E-AMRD
    Method: /G-AMRD
    Method: /eAMRw
    Method: /eAMRwc
    Method: /eMMRw
    Method: /eMMRwc
    Method: /gAMRw
    Method: /gAMRwc
    Method: /gMMRw
    Method: /gMMRwc
    Method: /AMRwc
    Method: /MMRwc
    Method: /eAMRd
    Method: /eAMRdc
    Method: /eMMRd
    Method: /eMMRdc
    Method: /gAMRd
    Method: /gAMRdc
    Method: /gMMRd
    Method: /gMMRdc
  WeightType: B
    Method: /EV
    Method: /GM
    Method: /WMIN
    Method: /DMIN
    Method: /MMRW
    Method: /MMRD
    Method: /E-MMRW
    Method: /G-MMRW
    Method: /E-MMRD
    Method: /G-MMRD
    Method: /AMRW
    Method: /AMRD
    Method: /E-AMRW
    Method: /G-AMRW
    Method: /E-AMRD
    Method: /G

Row,WeightType,Method,UtilityCase,UtilityIndex,RepeatIndex,sL_true,sR_true,s_mid_true,s_width_true,tL_est,tR_est,t_mid_est,t_width_est
,String,String,String,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,A,/EV,u1,1,1,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
2,A,/EV,u1,1,2,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
3,A,/EV,u1,1,3,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
4,A,/EV,u1,1,4,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
5,A,/EV,u1,1,5,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
6,A,/EV,u1,1,6,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
7,A,/EV,u1,1,7,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
8,A,/EV,u1,1,8,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0
9,A,/EV,u1,1,9,0.884956,1.14943,1.01719,0.26447,1.0,1.0,1.0,0.0


In [5]:
using CSV
using DataFrames
using Statistics

# ========== データ読み込み ==========

input_csv = "./results/trange_tidy/trange_tidy_u1u2_all_methods.csv"
df = CSV.read(input_csv, DataFrame)

println("Loaded size: ", size(df))

# ========== 誤差列の追加 ==========

# 中心・幅の誤差
df.mid_error   = df.t_mid_est   .- df.s_mid_true
df.width_error = df.t_width_est .- df.s_width_true

# 絶対誤差
df.abs_mid_error   = abs.(df.mid_error)
df.abs_width_error = abs.(df.width_error)

# 正解区間内に入っているか（お好みで）
df.mid_in_true_range = (df.t_mid_est .>= df.sL_true) .& (df.t_mid_est .<= df.sR_true)

# ========== 1. Method × UtilityCase ごとのまとめ表 ==========

group_cols = [:Method, :UtilityCase]

summary_by_method = combine(
    groupby(df, group_cols),
    :t_mid_est      => mean => :mean_t_mid_est,
    :t_mid_est      => std  => :std_t_mid_est,
    :t_width_est    => mean => :mean_t_width_est,
    :t_width_est    => std  => :std_t_width_est,
    :abs_mid_error  => mean => :mean_abs_mid_error,
    :abs_width_error=> mean => :mean_abs_width_error,
    :mid_in_true_range => x -> mean(x) => :ratio_mid_in_true_range,
)

println("\n=== summary_by_method (Method × UtilityCase) ===")
show(summary_by_method, allrows=false, allcols=true)

# ========== 2. WeightType も含めた細かい表 ==========

group_cols2 = [:WeightType, :Method, :UtilityCase]

summary_by_wt_method = combine(
    groupby(df, group_cols2),
    :t_width_est    => mean => :mean_t_width_est,
    :t_width_est    => std  => :std_t_width_est,
    :abs_mid_error  => mean => :mean_abs_mid_error,
    :abs_width_error=> mean => :mean_abs_width_error,
)

println("\n=== summary_by_wt_method (WeightType × Method × UtilityCase) ===")
show(summary_by_wt_method, allrows=false, allcols=true)

# ========== 3. ピボット的な見せ方（一例） ==========

# 例: UtilityCase=u1 だけに絞って、Method × WeightType の表にする
df_u1 = filter(row -> row.UtilityCase == "u1", df)

summary_u1 = combine(
    groupby(df_u1, [:Method, :WeightType]),
    :abs_mid_error => mean => :mean_abs_mid_error,
)

# Method を行、WeightType を列にしてピボット
pivot_u1 = unstack(summary_u1, :Method, :WeightType, :mean_abs_mid_error)

println("\n=== pivot_u1: mean_abs_mid_error (rows=Method, cols=WeightType) ===")
show(pivot_u1, allrows=false, allcols=true)

# ========== 4. 結果を CSV にも保存しておく（必要なら） ==========

mkpath("./results/trange_summary")

CSV.write("./results/trange_summary/summary_by_method.csv", summary_by_method)
CSV.write("./results/trange_summary/summary_by_wt_method.csv", summary_by_wt_method)
CSV.write("./results/trange_summary/pivot_u1_mean_abs_mid_error.csv", pivot_u1)

println("\nSaved summary tables to ./results/trange_summary/")


Loaded size: (3400000, 13)

=== summary_by_method (Method × UtilityCase) ===
68×9 DataFrame
 Row │ Method   UtilityCase  mean_t_mid_est  std_t_mid_est  mean_t_width_est  std_t_width_est  mean_abs_mid_error  mean_abs_width_error  mid_in_true_range_function      
     │ String7  String3      Float64         Float64        Float64           Float64          Float64             Float64               Pair{Float64, Symbol}           
─────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ /EV      u1                  1.0         6.75273e-7         0.0              0.0                0.0257186              0.320484   1.0=>:ratio_mid_in_true_range
   2 │ /EV      u2                  1.0         6.75273e-7         0.0              0.0                0.0257186              0.320484   1.0=>:ratio_mid_in_true_range
   3 │ /GM      u1                  1.0         7.85944e-7      

In [3]:
using StatsPlots  # Pkg.add("StatsPlots") しておく
default(fmt = :png)

"""
    plot_trange_hist2d(df; outdir="results/trange_plots")

真の中心 vs 推定の中心、真の幅 vs 推定の幅を手法ごとに 2Dヒストで出力。
"""
function plot_trange_hist2d(df::DataFrame; outdir="results/trange_plots")
    mkpath(outdir)

    for wt in unique(df.WeightType)
        for ucase in unique(df.UtilityCase)
            sub_wu = df[(df.WeightType .== wt) .& (df.UtilityCase .== ucase), :]

            for method in unique(sub_wu.Method)
                sub = sub_wu[sub_wu.Method .== method, :]

                # ① 中心の 2D ヒスト（真=縦, 推定=横）
                p1 = histogram2d(
                    sub.s_mid_true,
                    sub.t_mid_est;
                    bins=(30,30),
                    xlabel = "Estimated t mid",
                    ylabel = "True t mid",
                    title  = "Mid: $method  ($wt, $ucase)",
                    aspect_ratio = 1,
                    colorbar = true,
                )
                png(joinpath(outdir, "hist2d_mid_$(wt)_$(ucase)_$(replace(method, '/' => ""))"))

                # ② 幅の 2D ヒスト
                p2 = histogram2d(
                    sub.s_width_true,
                    sub.t_width_est;
                    bins=(30,30),
                    xlabel = "Estimated t width",
                    ylabel = "True t width",
                    title  = "Width: $method  ($wt, $ucase)",
                    aspect_ratio = 1,
                    colorbar = true,
                )
                png(joinpath(outdir, "hist2d_width_$(wt)_$(ucase)_$(replace(method, '/' => ""))"))
            end
        end
    end
end

# 実行例
plot_trange_hist2d(df_trange; outdir="results/trange_plots")


GKS: could not find font IPAexGothic.ttf
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ PlotUtils /root/.julia/packages/PlotUtils/dVEMd/src/ticks.jl:194
┌ Warning: No strict ticks found
└ @ Plot